# 🔐 AI-Based GitHub Security Vulnerability Scanner
### with Excel Reporting & AI-Powered Remediation

---

## 📌 Project Overview

This notebook provides a **complete end-to-end security vulnerability scanner** for GitHub repositories. It combines:

- **Static Code Analysis** – Regex-based detection of hardcoded secrets, insecure patterns
- **Dependency Scanning** – OSV API checks against known CVEs
- **AI-Powered Analysis** – Google Gemini (free tier) explains vulnerabilities and provides fixes
- **Risk Scoring** – Numeric scoring to classify overall repository risk
- **Excel Export** – Professional multi-sheet report with formatting
- **Visualizations** – Charts and dashboards inside the notebook

---

## 🏗️ Architecture Overview

```
┌─────────────────────────────────────────────────────────────────┐
│                    GitHub Security Scanner                       │
│                                                                  │
│  ┌──────────┐    ┌──────────────┐    ┌───────────────────────┐  │
│  │  GitHub   │───▶│  File Parser │───▶│  Vulnerability Engine │  │
│  │   API    │    │ .py .js .txt │    │ Secrets | Code | Deps │  │
│  └──────────┘    └──────────────┘    └──────────┬────────────┘  │
│                                                  │               │
│                                       ┌──────────▼────────────┐  │
│                                       │    AI Analysis         │  │
│                                       │  (Google Gemini API)   │  │
│                                       │  Severity | Fix | Code │  │
│                                       └──────────┬────────────┘  │
│                                                  │               │
│              ┌──────────────┐        ┌──────────▼────────────┐  │
│              │ Visualizations│◀───────│   Risk Scoring Engine │  │
│              │ Charts + Dash │        │  Score | Classification│  │
│              └──────────────┘        └──────────┬────────────┘  │
│                                                  │               │
│                                       ┌──────────▼────────────┐  │
│                                       │    Excel Report        │  │
│                                       │  Summary + Findings    │  │
│                                       └───────────────────────┘  │
└─────────────────────────────────────────────────────────────────┘
```

---

## 📋 Setup Requirements

**APIs Needed (all free):**
- `GITHUB_TOKEN` – Optional, but recommended (get at https://github.com/settings/tokens)
- `GEMINI_API_KEY` – Free at https://aistudio.google.com/app/apikey (generous free tier)
- `OSV API` – No key needed (free, public)

**Test Repository (safe to scan):** `https://github.com/bridgecrewio/terragoat`

## 📦 Section 1 — Install Dependencies

In [ ]:
# Install all required packages
import subprocess, sys

packages = [
    'PyGithub',
    'requests',
    'pandas',
    'openpyxl',
    'matplotlib',
    'seaborn',
    'tqdm',
    'google-generativeai',
    'packaging'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    print(f'  ✅ {pkg} installed')

print('\n🎉 All dependencies installed successfully!')

## ⚙️ Section 2 — Imports & Global Configuration

In [ ]:
import re
import os
import json
import time
import base64
import warnings
import requests
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime
from tqdm.notebook import tqdm
from github import Github, GithubException
from openpyxl import Workbook
from openpyxl.styles import (
    Font, PatternFill, Alignment, Border, Side, GradientFill
)
from openpyxl.utils import get_column_letter
from openpyxl.chart import BarChart, PieChart, Reference
from packaging.version import Version, InvalidVersion
import google.generativeai as genai

warnings.filterwarnings('ignore')
matplotlib.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans'})

# ─── Global Settings ──────────────────────────────────────────────
SEVERITY_SCORE_MAP  = {'Low': 2, 'Medium': 5, 'High': 8, 'Critical': 10}
SEVERITY_COLOR_MAP  = {'Low': '#27AE60', 'Medium': '#F39C12', 'High': '#E74C3C', 'Critical': '#8E44AD'}
RISK_THRESHOLDS     = {'Safe': 20, 'Moderate Risk': 60, 'High Risk': 120}  # > 120 = Critical Risk
OSV_API_URL         = 'https://api.osv.dev/v1/query'
OUTPUT_EXCEL_FILE   = 'github_security_vulnerability_report.xlsx'
MAX_FILES_TO_SCAN   = 100   # Cap to respect rate limits
AI_DELAY_SECONDS    = 2     # Delay between AI calls to avoid quota issues

print('✅ Imports loaded and configuration set.')
print(f'   Output file: {OUTPUT_EXCEL_FILE}')

## 🔑 Section 3 — API Keys & GitHub Authentication

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  ✏️  FILL IN YOUR CREDENTIALS BELOW
# ─────────────────────────────────────────────────────────────────

GITHUB_REPO_URL = 'https://github.com/OWASP/WebGoat'   # 🔁 Change to any public repo
GITHUB_TOKEN    = ''       # Optional — paste token or leave blank
GEMINI_API_KEY  = ''       # Required — get free key at aistudio.google.com

# ─────────────────────────────────────────────────────────────────

def authenticate_github(token: str = '') -> Github:
    """Authenticate to GitHub API (with or without token)."""
    if token.strip():
        g = Github(token.strip())
        user = g.get_user().login
        rate = g.get_rate_limit().core
        print(f'✅ GitHub authenticated as: {user}')
        print(f'   Rate limit: {rate.remaining}/{rate.limit} remaining')
    else:
        g = Github()
        rate = g.get_rate_limit().core
        print(f'⚠️  GitHub unauthenticated (60 req/hour). Token recommended.')
        print(f'   Rate limit: {rate.remaining}/{rate.limit} remaining')
    return g


def configure_gemini(api_key: str):
    """Configure Google Gemini AI."""
    if not api_key.strip():
        print('⚠️  No Gemini API key provided. AI analysis will be SKIPPED.')
        print('   Get a free key at: https://aistudio.google.com/app/apikey')
        return None
    genai.configure(api_key=api_key.strip())
    model = genai.GenerativeModel('gemini-1.5-flash')  # Free tier model
    print('✅ Gemini AI configured (gemini-1.5-flash - free tier)')
    return model


def parse_repo_url(url: str) -> str:
    """Extract 'owner/repo' from a GitHub URL."""
    url = url.rstrip('/').replace('.git', '')
    parts = url.split('github.com/')
    if len(parts) < 2:
        raise ValueError(f'Invalid GitHub URL: {url}')
    return parts[1].strip()


# Run authentication
github_client = authenticate_github(GITHUB_TOKEN)
gemini_model  = configure_gemini(GEMINI_API_KEY)
repo_path     = parse_repo_url(GITHUB_REPO_URL)
print(f'\n📂 Repository target: {repo_path}')

## 📥 Section 4 — Repository Fetching & File Parsing

In [ ]:
# File extensions to scan
SCAN_EXTENSIONS = {
    '.py', '.js', '.ts', '.jsx', '.tsx', '.java', '.go',
    '.rb', '.php', '.cs', '.cpp', '.c', '.sh', '.yaml',
    '.yml', '.env', '.cfg', '.conf', '.ini', '.xml', '.json'
}
DEPENDENCY_FILES = {
    'requirements.txt', 'package.json', 'Pipfile',
    'pyproject.toml', 'setup.py', 'composer.json',
    'Gemfile', 'pom.xml', 'build.gradle'
}


def fetch_repository_files(g: Github, repo_name: str) -> dict:
    """
    Fetch all scannable files from a GitHub repository.
    Returns dict: {'code_files': [...], 'dependency_files': [...], 'repo_meta': {...}}
    """
    print(f'\n🔍 Fetching repository: {repo_name}')
    try:
        repo = g.get_repo(repo_name)
    except GithubException as e:
        raise RuntimeError(f'Cannot access repo "{repo_name}": {e.data.get("message", str(e))}')

    meta = {
        'name'        : repo.name,
        'full_name'   : repo.full_name,
        'url'         : repo.html_url,
        'description' : repo.description or 'N/A',
        'language'    : repo.language or 'N/A',
        'stars'       : repo.stargazers_count,
        'forks'       : repo.forks_count,
        'size_kb'     : repo.size,
        'scanned_at'  : datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    print(f'   📌 {repo.full_name} | ⭐ {repo.stargazers_count} stars | Language: {repo.language}')

    code_files  = []
    dep_files   = []
    file_count  = 0

    def traverse(contents, depth=0):
        nonlocal file_count
        if file_count >= MAX_FILES_TO_SCAN:
            return
        for item in contents:
            if file_count >= MAX_FILES_TO_SCAN:
                break
            if item.type == 'dir' and depth < 4:
                try:
                    traverse(repo.get_contents(item.path), depth + 1)
                except GithubException:
                    pass
            elif item.type == 'file':
                ext  = os.path.splitext(item.name)[1].lower()
                name = item.name.lower()
                if name in DEPENDENCY_FILES:
                    dep_files.append(item)
                    file_count += 1
                elif ext in SCAN_EXTENSIONS and item.size < 500_000:  # skip huge files
                    code_files.append(item)
                    file_count += 1

    try:
        root_contents = repo.get_contents('')
        traverse(root_contents)
    except GithubException as e:
        print(f'⚠️  Partial traversal: {e}')

    print(f'   📄 Code files found    : {len(code_files)}')
    print(f'   📦 Dependency files    : {len(dep_files)}')
    print(f'   📊 Total files fetched : {file_count} (cap: {MAX_FILES_TO_SCAN})')

    return {'code_files': code_files, 'dependency_files': dep_files, 'repo_meta': meta, 'repo': repo}


def decode_file_content(file_item) -> str:
    """Safely decode a GitHub file's content."""
    try:
        raw = file_item.content
        if file_item.encoding == 'base64':
            return base64.b64decode(raw).decode('utf-8', errors='replace')
        return raw or ''
    except Exception:
        return ''


# Run repository fetch
repo_data = fetch_repository_files(github_client, repo_path)

## 🔎 Section 5 — Vulnerability Detection Engine

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  A) HARDCODED SECRETS PATTERNS
# ═══════════════════════════════════════════════════════════════

SECRET_PATTERNS = [
    {
        'name'   : 'AWS Access Key',
        'regex'  : r'(?<![A-Z0-9])AKIA[0-9A-Z]{16}(?![A-Z0-9])',
        'severity': 'Critical',
        'type'   : 'Hardcoded Secret',
        'ref'    : 'https://docs.aws.amazon.com/general/latest/gr/aws-security-credentials.html'
    },
    {
        'name'   : 'AWS Secret Key',
        'regex'  : r'(?i)aws.{0,20}secret.{0,20}[=:\s][\'"]?[A-Za-z0-9/+]{40}',
        'severity': 'Critical',
        'type'   : 'Hardcoded Secret',
        'ref'    : 'https://docs.aws.amazon.com/general/latest/gr/aws-security-credentials.html'
    },
    {
        'name'   : 'Generic API Key',
        'regex'  : r'(?i)(api[_-]?key|apikey|api[_-]?secret)\s*[=:>]\s*[\'"]([A-Za-z0-9\-_]{20,60})[\'"]',
        'severity': 'High',
        'type'   : 'Hardcoded Secret',
        'ref'    : 'https://owasp.org/www-community/vulnerabilities/Use_of_hard-coded_credentials'
    },
    {
        'name'   : 'Hardcoded Password',
        'regex'  : r'(?i)(password|passwd|pwd)\s*[=:>]\s*[\'"]([^\s\'"]{6,})[\'"]',
        'severity': 'High',
        'type'   : 'Hardcoded Secret',
        'ref'    : 'https://cwe.mitre.org/data/definitions/259.html'
    },
    {
        'name'   : 'GitHub Token',
        'regex'  : r'gh[pousr]_[A-Za-z0-9_]{36,255}',
        'severity': 'Critical',
        'type'   : 'Hardcoded Secret',
        'ref'    : 'https://docs.github.com/en/authentication/keeping-your-account-and-data-secure'
    },
    {
        'name'   : 'Slack Token',
        'regex'  : r'xox[baprs]-[0-9A-Za-z]{10,48}',
        'severity': 'High',
        'type'   : 'Hardcoded Secret',
        'ref'    : 'https://api.slack.com/authentication/token-types'
    },
    {
        'name'   : 'Private RSA Key',
        'regex'  : r'-----BEGIN (RSA |EC )?PRIVATE KEY-----',
        'severity': 'Critical',
        'type'   : 'Hardcoded Secret',
        'ref'    : 'https://cwe.mitre.org/data/definitions/321.html'
    },
    {
        'name'   : 'Generic Secret',
        'regex'  : r'(?i)(secret[_-]?key|client[_-]?secret)\s*[=:>]\s*[\'"]([A-Za-z0-9\-_]{16,80})[\'"]',
        'severity': 'High',
        'type'   : 'Hardcoded Secret',
        'ref'    : 'https://owasp.org/www-community/vulnerabilities/Use_of_hard-coded_credentials'
    },
    {
        'name'   : 'Database Connection String',
        'regex'  : r'(?i)(mongodb|postgresql|mysql|redis):\/\/[^\s"\'>]+:[^\s"\'>@]+@',
        'severity': 'Critical',
        'type'   : 'Hardcoded Secret',
        'ref'    : 'https://owasp.org/www-community/vulnerabilities/Use_of_hard-coded_credentials'
    },
]

# ═══════════════════════════════════════════════════════════════
#  B) INSECURE CODE PATTERNS
# ═══════════════════════════════════════════════════════════════

INSECURE_CODE_PATTERNS = [
    {
        'name'   : 'Use of eval()',
        'regex'  : r'\beval\s*\(',
        'severity': 'High',
        'type'   : 'Insecure Code',
        'ref'    : 'https://owasp.org/www-community/attacks/Code_Injection'
    },
    {
        'name'   : 'Use of exec()',
        'regex'  : r'\bexec\s*\(',
        'severity': 'High',
        'type'   : 'Insecure Code',
        'ref'    : 'https://cwe.mitre.org/data/definitions/78.html'
    },
    {
        'name'   : 'subprocess shell=True',
        'regex'  : r'subprocess\.[a-zA-Z_]+\([^)]*shell\s*=\s*True',
        'severity': 'High',
        'type'   : 'Insecure Code',
        'ref'    : 'https://docs.python.org/3/library/subprocess.html#security-considerations'
    },
    {
        'name'   : 'pickle.loads Usage',
        'regex'  : r'pickle\.loads?\s*\(',
        'severity': 'Critical',
        'type'   : 'Insecure Deserialization',
        'ref'    : 'https://docs.python.org/3/library/pickle.html#security'
    },
    {
        'name'   : 'MD5 Hashing',
        'regex'  : r'(?i)hashlib\.md5\s*\(',
        'severity': 'Medium',
        'type'   : 'Weak Cryptography',
        'ref'    : 'https://www.nist.gov/publications/nist-policy-cryptographic-hash-functions'
    },
    {
        'name'   : 'SHA1 Hashing',
        'regex'  : r'(?i)hashlib\.sha1\s*\(',
        'severity': 'Medium',
        'type'   : 'Weak Cryptography',
        'ref'    : 'https://shattered.io/'
    },
    {
        'name'   : 'Flask Debug Mode',
        'regex'  : r'(?i)app\.run\s*\([^)]*debug\s*=\s*True',
        'severity': 'High',
        'type'   : 'Misconfiguration',
        'ref'    : 'https://flask.palletsprojects.com/en/latest/debugging/'
    },
    {
        'name'   : 'SQL Injection Risk (string format)',
        'regex'  : r'(?i)(execute|cursor\.execute)\s*\([^)]*(%s|\{|format|f\")',
        'severity': 'Critical',
        'type'   : 'Injection',
        'ref'    : 'https://owasp.org/www-community/attacks/SQL_Injection'
    },
    {
        'name'   : 'SSL Verification Disabled',
        'regex'  : r'verify\s*=\s*False',
        'severity': 'High',
        'type'   : 'Insecure Transport',
        'ref'    : 'https://owasp.org/www-community/attacks/Man-in-the-middle_attack'
    },
    {
        'name'   : 'Random Seed Fixed',
        'regex'  : r'random\.seed\s*\(\s*[0-9]',
        'severity': 'Low',
        'type'   : 'Weak Randomness',
        'ref'    : 'https://cwe.mitre.org/data/definitions/335.html'
    },
    {
        'name'   : 'Insecure Deserialization (yaml.load)',
        'regex'  : r'yaml\.load\s*\([^)]*\)',
        'severity': 'High',
        'type'   : 'Insecure Deserialization',
        'ref'    : 'https://pyyaml.org/wiki/PyYAMLDocumentation'
    },
    {
        'name'   : 'Hardcoded IP Address',
        'regex'  : r'(?<!\.)(\b(?:127\.0\.0\.1|0\.0\.0\.0|192\.168\.[0-9]{1,3}\.[0-9]{1,3})\b)(?!\.)',
        'severity': 'Low',
        'type'   : 'Misconfiguration',
        'ref'    : 'https://cwe.mitre.org/data/definitions/1219.html'
    },
    {
        'name'   : 'Open CORS Policy',
        'regex'  : r'(?i)(cors|Access-Control-Allow-Origin).*[\*]',
        'severity': 'Medium',
        'type'   : 'Misconfiguration',
        'ref'    : 'https://owasp.org/www-project-web-security-testing-guide/v42/4-Web_Application_Security_Testing/11-Client_Side_Testing/07-Testing_Cross_Origin_Resource_Sharing'
    },
    {
        'name'   : 'Command Injection Risk (os.system)',
        'regex'  : r'os\.system\s*\(',
        'severity': 'High',
        'type'   : 'Injection',
        'ref'    : 'https://cwe.mitre.org/data/definitions/78.html'
    },
    {
        'name'   : 'Weak DES/3DES Cipher',
        'regex'  : r'(?i)(DES|3DES|TripleDES|Blowfish)[\.\(]',
        'severity': 'Medium',
        'type'   : 'Weak Cryptography',
        'ref'    : 'https://cwe.mitre.org/data/definitions/327.html'
    },
]


def scan_code_file(file_item, repo_name: str) -> list:
    """
    Scan a single source code file for security vulnerabilities.
    Returns a list of vulnerability dictionaries.
    """
    content = decode_file_content(file_item)
    if not content:
        return []

    lines   = content.splitlines()
    vulns   = []
    all_patterns = SECRET_PATTERNS + INSECURE_CODE_PATTERNS

    for pattern in all_patterns:
        regex = re.compile(pattern['regex'], re.IGNORECASE if '(?i)' not in pattern['regex'] else 0)
        # Match per line (with line numbers)
        for line_num, line in enumerate(lines, start=1):
            stripped = line.strip()
            if stripped.startswith('#') or stripped.startswith('//'):
                continue  # Skip comment lines
            match = regex.search(line)
            if match:
                # Redact matched secret value for safety
                snippet = line[:120].strip()
                if pattern['type'] == 'Hardcoded Secret':
                    snippet = re.sub(r'([\'"])[A-Za-z0-9\-_/+]{8,}[\'"]', r'\1[REDACTED]\1', snippet)

                vulns.append({
                    'Repository Name'  : repo_name,
                    'File Name'        : file_item.path,
                    'Line Number'      : line_num,
                    'Vulnerability Type': pattern['type'],
                    'Vulnerability Name': pattern['name'],
                    'Severity'         : pattern['severity'],
                    'Risk Score'       : SEVERITY_SCORE_MAP.get(pattern['severity'], 1),
                    'Code Snippet'     : snippet,
                    'Reference Link'   : pattern['ref'],
                    'Description'      : '',   # filled by AI
                    'AI Explanation'   : '',
                    'Recommended Fix'  : '',
                    'Secure Code Example': ''
                })
                break  # One finding per pattern per file
    return vulns


print('✅ Vulnerability patterns loaded.')
print(f'   Secret patterns     : {len(SECRET_PATTERNS)}')
print(f'   Insecure patterns   : {len(INSECURE_CODE_PATTERNS)}')
print(f'   Total patterns      : {len(SECRET_PATTERNS) + len(INSECURE_CODE_PATTERNS)}')

## 📦 Section 6 — Dependency Vulnerability Scanner (OSV API)

In [ ]:
def parse_requirements_txt(content: str) -> list:
    """Parse requirements.txt into list of {name, version} dicts."""
    packages = []
    for line in content.splitlines():
        line = line.strip()
        if not line or line.startswith('#') or line.startswith('-'):
            continue
        # Handle: Flask==2.0.1, requests>=2.26.0, numpy~=1.21
        match = re.match(r'^([A-Za-z0-9_\-\.\[\]]+)\s*([=><~!]=?)\s*([0-9][^\s,;]*)?', line)
        if match:
            name    = match.group(1).split('[')[0]  # strip extras
            version = match.group(3) or ''
            packages.append({'name': name, 'version': version.strip()})
    return packages


def parse_package_json(content: str) -> list:
    """Parse package.json into list of {name, version} dicts."""
    packages = []
    try:
        data = json.loads(content)
        all_deps = {}
        all_deps.update(data.get('dependencies', {}))
        all_deps.update(data.get('devDependencies', {}))
        for name, ver in all_deps.items():
            clean_ver = re.sub(r'^[^0-9]*', '', ver)  # remove ^, ~, etc.
            packages.append({'name': name, 'version': clean_ver})
    except json.JSONDecodeError:
        pass
    return packages


def check_osv_vulnerability(package_name: str, version: str, ecosystem: str) -> list:
    """
    Query the OSV API (free, no key required) for known vulnerabilities.
    Returns list of CVE/vulnerability dicts.
    """
    payload = {
        'package': {'name': package_name, 'ecosystem': ecosystem}
    }
    if version:
        payload['version'] = version

    try:
        response = requests.post(OSV_API_URL, json=payload, timeout=10)
        if response.status_code == 200:
            data = response.json()
            return data.get('vulns', [])
    except requests.RequestException:
        pass
    return []


def scan_dependency_file(file_item, repo_name: str) -> list:
    """
    Parse a dependency file and check each package against the OSV API.
    Returns vulnerability list.
    """
    content   = decode_file_content(file_item)
    file_name = file_item.name.lower()
    vulns     = []

    if file_name == 'requirements.txt':
        packages  = parse_requirements_txt(content)
        ecosystem = 'PyPI'
    elif file_name == 'package.json':
        packages  = parse_package_json(content)
        ecosystem = 'npm'
    else:
        return []

    for pkg in tqdm(packages, desc=f'Scanning {file_name}', leave=False):
        osv_hits = check_osv_vulnerability(pkg['name'], pkg['version'], ecosystem)
        for hit in osv_hits[:2]:  # Cap at 2 CVEs per package
            severity = 'Medium'
            cvss_score = 0.0
            cve_id = ''

            # Extract severity from CVSS
            for alias in hit.get('aliases', []):
                if alias.startswith('CVE-'):
                    cve_id = alias
            for sev in hit.get('severity', []):
                score_str = sev.get('score', '')
                nums = re.findall(r'[0-9]+\.?[0-9]*', score_str)
                if nums:
                    cvss_score = float(nums[0])
            if cvss_score >= 9.0:
                severity = 'Critical'
            elif cvss_score >= 7.0:
                severity = 'High'
            elif cvss_score >= 4.0:
                severity = 'Medium'
            else:
                severity = 'Low'

            summary = hit.get('summary', 'Known vulnerability in this package version.')
            fix_ver = ''
            for affected in hit.get('affected', []):
                for rng in affected.get('ranges', []):
                    for ev in rng.get('events', []):
                        if 'fixed' in ev:
                            fix_ver = ev['fixed']

            vulns.append({
                'Repository Name'   : repo_name,
                'File Name'         : file_item.path,
                'Line Number'       : 'N/A',
                'Vulnerability Type': 'Dependency Vulnerability',
                'Vulnerability Name': f'{pkg["name"]} {pkg["version"]} — {cve_id or hit.get("id", "")}',
                'Severity'          : severity,
                'Risk Score'        : SEVERITY_SCORE_MAP.get(severity, 1),
                'Code Snippet'      : f'{pkg["name"]}=={pkg["version"]}',
                'Reference Link'    : f'https://osv.dev/vulnerability/{hit.get("id", "")}',
                'Description'       : summary,
                'AI Explanation'    : '',
                'Recommended Fix'   : f'Upgrade to version {fix_ver}' if fix_ver else 'Check latest safe version',
                'Secure Code Example': f'{pkg["name"]}>={fix_ver}' if fix_ver else f'# Upgrade {pkg["name"]}'
            })

        time.sleep(0.1)  # Respect OSV rate limits

    return vulns


print('✅ Dependency scanner (OSV API) ready.')

## 🤖 Section 7 — AI Analysis Integration (Google Gemini)

In [ ]:
AI_PROMPT_TEMPLATE = """\
You are a senior application security engineer. Analyze this security vulnerability and respond in JSON.

Vulnerability: {name}
Type: {type}
Detected In: {file}
Code Snippet: {snippet}
Severity: {severity}

Respond with ONLY a valid JSON object (no markdown, no explanation outside JSON):
{{
  "description": "<1-2 sentence description of what this vulnerability is and why it occurs>",
  "explanation": "<2-3 sentence explanation of the security danger, possible exploit scenario>",
  "severity": "<Low|Medium|High|Critical — may revise based on context>",
  "remediation": "<2-3 concrete steps to fix this>",
  "secure_code": "<a short secure code example (5-10 lines max) replacing the vulnerable pattern>"
}}
"""

_ai_fallback_cache = {}  # Cache to avoid duplicate AI calls


def get_ai_analysis(model, vuln: dict) -> dict:
    """
    Call Gemini AI to analyze a vulnerability.
    Returns dict with description, explanation, severity, remediation, secure_code.
    Falls back to rule-based values if AI unavailable.
    """
    fallback = {
        'description' : vuln.get('Description') or f'{vuln["Vulnerability Name"]} detected in source code.',
        'explanation' : 'Manual review required. This pattern may indicate a security risk.',
        'severity'    : vuln['Severity'],
        'remediation' : vuln.get('Recommended Fix') or 'Follow OWASP best practices.',
        'secure_code' : '# Review and refactor this code following secure coding guidelines'
    }

    if model is None:
        return fallback

    # Use cache key to avoid duplicate calls
    cache_key = f"{vuln['Vulnerability Name']}|{vuln['Vulnerability Type']}"
    if cache_key in _ai_fallback_cache:
        return _ai_fallback_cache[cache_key]

    prompt = AI_PROMPT_TEMPLATE.format(
        name    = vuln['Vulnerability Name'],
        type    = vuln['Vulnerability Type'],
        file    = vuln['File Name'],
        snippet = vuln.get('Code Snippet', '')[:300],
        severity= vuln['Severity']
    )

    try:
        time.sleep(AI_DELAY_SECONDS)
        response = model.generate_content(prompt)
        raw_text = response.text.strip()

        # Strip potential markdown code fences
        raw_text = re.sub(r'^```(?:json)?\s*', '', raw_text)
        raw_text = re.sub(r'\s*```$', '', raw_text)

        parsed   = json.loads(raw_text)
        result   = {
            'description' : parsed.get('description', fallback['description']),
            'explanation' : parsed.get('explanation', fallback['explanation']),
            'severity'    : parsed.get('severity', vuln['Severity']),
            'remediation' : parsed.get('remediation', fallback['remediation']),
            'secure_code' : parsed.get('secure_code', fallback['secure_code'])
        }
        _ai_fallback_cache[cache_key] = result
        return result

    except json.JSONDecodeError:
        # Gemini returned non-JSON — extract what we can
        raw = response.text if hasattr(response, 'text') else ''
        fallback['explanation'] = raw[:400] if raw else fallback['explanation']
        return fallback
    except Exception as e:
        print(f'   ⚠️  AI error ({type(e).__name__}): {str(e)[:80]}')
        return fallback


print('✅ AI analysis module ready.')
if gemini_model is None:
    print('   Mode: Rule-based fallback (no API key)')
else:
    print('   Mode: Google Gemini 1.5 Flash (free tier)')

## ▶️ Section 8 — Run the Full Scan

In [ ]:
def run_full_scan(repo_data: dict, gemini_model) -> pd.DataFrame:
    """
    Orchestrate the complete vulnerability scan pipeline.
    Returns a pandas DataFrame of all findings.
    """
    repo_name = repo_data['repo_meta']['full_name']
    all_vulns = []

    print('\n' + '═'*65)
    print(f'  🚀 STARTING SECURITY SCAN: {repo_name}')
    print('═'*65)

    # ── Phase 1: Code scanning ──────────────────────────────────────
    print('\n📋 Phase 1/3 — Scanning source code files...')
    code_files = repo_data['code_files']
    for file_item in tqdm(code_files, desc='Code Scan', unit='file'):
        vulns = scan_code_file(file_item, repo_name)
        all_vulns.extend(vulns)

    print(f'   Found {len(all_vulns)} code vulnerabilities')

    # ── Phase 2: Dependency scanning ────────────────────────────────
    print('\n📦 Phase 2/3 — Scanning dependency files (OSV API)...')
    dep_files = repo_data['dependency_files']
    dep_vulns_count = 0
    for file_item in dep_files:
        dep_vulns = scan_dependency_file(file_item, repo_name)
        all_vulns.extend(dep_vulns)
        dep_vulns_count += len(dep_vulns)
    print(f'   Found {dep_vulns_count} dependency vulnerabilities')

    if not all_vulns:
        print('\n✅ No vulnerabilities detected!')
        return pd.DataFrame()

    # ── Phase 3: AI analysis ────────────────────────────────────────
    print(f'\n🤖 Phase 3/3 — Running AI analysis on {len(all_vulns)} findings...')
    for vuln in tqdm(all_vulns, desc='AI Analysis', unit='vuln'):
        ai = get_ai_analysis(gemini_model, vuln)
        vuln['Description']       = ai['description']
        vuln['AI Explanation']    = ai['explanation']
        vuln['Severity']          = ai['severity']           # AI may revise
        vuln['Risk Score']        = SEVERITY_SCORE_MAP.get(ai['severity'], vuln['Risk Score'])
        vuln['Recommended Fix']   = ai['remediation']
        vuln['Secure Code Example'] = ai['secure_code']

    # ── Build DataFrame ──────────────────────────────────────────────
    columns = [
        'Repository Name', 'File Name', 'Line Number', 'Vulnerability Type',
        'Vulnerability Name', 'Severity', 'Risk Score', 'Code Snippet',
        'Description', 'AI Explanation', 'Recommended Fix',
        'Secure Code Example', 'Reference Link'
    ]
    df = pd.DataFrame(all_vulns, columns=columns)

    # Sort by severity
    sev_order = {'Critical': 0, 'High': 1, 'Medium': 2, 'Low': 3}
    df['_sev_order'] = df['Severity'].map(sev_order).fillna(4)
    df = df.sort_values('_sev_order').drop(columns='_sev_order').reset_index(drop=True)

    print('\n' + '═'*65)
    print(f'  ✅ SCAN COMPLETE — {len(df)} total vulnerabilities found')
    print('═'*65)
    print(df['Severity'].value_counts().to_string())

    return df


# ── Execute the scan ────────────────────────────────────────────────
df_vulns = run_full_scan(repo_data, gemini_model)

## 📊 Section 9 — Risk Scoring Engine

In [ ]:
def calculate_risk_score(df: pd.DataFrame, repo_meta: dict) -> dict:
    """
    Calculate a total risk score and repository risk classification.
    Returns a risk_report dict.
    """
    if df.empty:
        return {
            'total_score'       : 0,
            'classification'    : 'Safe',
            'classification_color': SEVERITY_COLOR_MAP['Low'],
            'counts'            : {},
            'type_counts'       : {},
            'top_files'         : [],
            'repo_meta'         : repo_meta
        }

    total_score = int(df['Risk Score'].sum())

    if total_score <= RISK_THRESHOLDS['Safe']:
        classification = 'Safe'
        cls_color = '#27AE60'
    elif total_score <= RISK_THRESHOLDS['Moderate Risk']:
        classification = 'Moderate Risk'
        cls_color = '#F39C12'
    elif total_score <= RISK_THRESHOLDS['High Risk']:
        classification = 'High Risk'
        cls_color = '#E74C3C'
    else:
        classification = 'Critical Risk'
        cls_color = '#8E44AD'

    counts     = df['Severity'].value_counts().to_dict()
    type_counts = df['Vulnerability Type'].value_counts().to_dict()
    top_files  = df['File Name'].value_counts().head(5).to_dict()

    report = {
        'total_score'         : total_score,
        'classification'      : classification,
        'classification_color': cls_color,
        'counts'              : counts,
        'type_counts'         : type_counts,
        'top_files'           : top_files,
        'repo_meta'           : repo_meta
    }

    print('\n' + '─'*55)
    print('  📊 RISK SCORING REPORT')
    print('─'*55)
    print(f'  Repository   : {repo_meta["full_name"]}')
    print(f'  Total Score  : {total_score}')
    print(f'  Classification: ⚠️  {classification}')
    print(f'  Breakdown    :')
    for sev in ['Critical', 'High', 'Medium', 'Low']:
        cnt = counts.get(sev, 0)
        bar = '█' * min(cnt, 30)
        print(f'    {sev:<10}: {bar} {cnt}')
    print('─'*55)

    return report


risk_report = calculate_risk_score(df_vulns, repo_data['repo_meta'])

## 📈 Section 10 — Visualizations

In [ ]:
def create_visualizations(df: pd.DataFrame, risk_report: dict):
    """Generate all charts and risk dashboard inside the notebook."""

    if df.empty:
        print('ℹ️  No vulnerabilities found — nothing to chart.')
        return

    sev_order   = ['Critical', 'High', 'Medium', 'Low']
    counts      = risk_report['counts']
    type_counts = risk_report['type_counts']

    # ── Severity bar colors ─────────────────────────────────────────
    colors = [SEVERITY_COLOR_MAP.get(s, '#95A5A6') for s in sev_order if s in counts]
    sev_present = [s for s in sev_order if s in counts]
    sev_values  = [counts[s] for s in sev_present]

    fig = plt.figure(figsize=(20, 16))
    fig.patch.set_facecolor('#0D1117')

    # ── Title ──────────────────────────────────────────────────────
    fig.suptitle(
        f'🔐 Security Scan Report — {risk_report["repo_meta"]["full_name"]}',
        fontsize=16, fontweight='bold', color='white', y=0.98
    )

    # ── 1. Severity Bar Chart ─────────────────────────────────────
    ax1 = fig.add_subplot(2, 3, 1)
    ax1.set_facecolor('#161B22')
    bars = ax1.bar(sev_present, sev_values, color=colors, edgecolor='white', linewidth=0.5)
    ax1.set_title('Vulnerabilities by Severity', color='white', fontweight='bold', pad=10)
    ax1.set_xlabel('Severity', color='#8B949E')
    ax1.set_ylabel('Count', color='#8B949E')
    ax1.tick_params(colors='white')
    ax1.spines[:].set_color('#30363D')
    for bar, val in zip(bars, sev_values):
        ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
                 str(val), ha='center', va='bottom', color='white', fontweight='bold')

    # ── 2. Severity Pie Chart ─────────────────────────────────────
    ax2 = fig.add_subplot(2, 3, 2)
    ax2.set_facecolor('#161B22')
    wedges, texts, autotexts = ax2.pie(
        sev_values, labels=sev_present, colors=colors,
        autopct='%1.1f%%', startangle=90,
        wedgeprops={'edgecolor': '#0D1117', 'linewidth': 2}
    )
    for text in texts + autotexts:
        text.set_color('white')
    ax2.set_title('Severity Distribution', color='white', fontweight='bold', pad=10)

    # ── 3. Vulnerability Type Bar Chart ──────────────────────────
    ax3 = fig.add_subplot(2, 3, 3)
    ax3.set_facecolor('#161B22')
    types   = list(type_counts.keys())[:8]
    t_vals  = [type_counts[t] for t in types]
    t_colors = plt.cm.Set3(np.linspace(0, 1, len(types)))
    h_bars  = ax3.barh(types, t_vals, color=t_colors, edgecolor='white', linewidth=0.5)
    ax3.set_title('Vulnerabilities by Category', color='white', fontweight='bold', pad=10)
    ax3.set_xlabel('Count', color='#8B949E')
    ax3.tick_params(colors='white')
    ax3.spines[:].set_color('#30363D')
    for bar, val in zip(h_bars, t_vals):
        ax3.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2.,
                 str(val), va='center', color='white', fontsize=9)

    # ── 4. Top Vulnerable Files ───────────────────────────────────
    ax4 = fig.add_subplot(2, 3, 4)
    ax4.set_facecolor('#161B22')
    top_files = risk_report['top_files']
    if top_files:
        f_names = [os.path.basename(f) for f in list(top_files.keys())]
        f_vals  = list(top_files.values())
        ax4.barh(f_names, f_vals, color='#58A6FF', edgecolor='white', linewidth=0.5)
        ax4.set_title('Top 5 Vulnerable Files', color='white', fontweight='bold', pad=10)
        ax4.set_xlabel('Vulnerability Count', color='#8B949E')
        ax4.tick_params(colors='white')
        ax4.spines[:].set_color('#30363D')

    # ── 5. Risk Score Gauge ───────────────────────────────────────
    ax5 = fig.add_subplot(2, 3, 5)
    ax5.set_facecolor('#161B22')
    ax5.set_xlim(0, 1)
    ax5.set_ylim(0, 1)
    ax5.axis('off')

    cls   = risk_report['classification']
    score = risk_report['total_score']
    cls_color = risk_report['classification_color']

    # Draw circular gauge
    circle_bg  = plt.Circle((0.5, 0.45), 0.35, color='#21262D', linewidth=3, fill=True)
    circle_ring= plt.Circle((0.5, 0.45), 0.35, color=cls_color, linewidth=8, fill=False)
    ax5.add_patch(circle_bg)
    ax5.add_patch(circle_ring)
    ax5.text(0.5, 0.52, str(score), ha='center', va='center',
             fontsize=36, fontweight='bold', color=cls_color)
    ax5.text(0.5, 0.38, 'RISK SCORE', ha='center', va='center',
             fontsize=10, color='#8B949E')
    ax5.text(0.5, 0.12, cls, ha='center', va='center',
             fontsize=14, fontweight='bold', color=cls_color,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='#21262D', edgecolor=cls_color, linewidth=2))
    ax5.set_title('Repository Risk Status', color='white', fontweight='bold', pad=10)

    # ── 6. Severity Stacked by Type ───────────────────────────────
    ax6 = fig.add_subplot(2, 3, 6)
    ax6.set_facecolor('#161B22')
    pivot = df.groupby(['Vulnerability Type', 'Severity']).size().unstack(fill_value=0)
    for sev in sev_order:
        if sev not in pivot.columns:
            pivot[sev] = 0
    pivot = pivot[sev_order]
    bottom = np.zeros(len(pivot))
    type_labels = [t[:20] for t in pivot.index]
    for sev in sev_order:
        if sev in pivot.columns:
            vals = pivot[sev].values
            ax6.bar(type_labels, vals, bottom=bottom,
                    label=sev, color=SEVERITY_COLOR_MAP[sev], edgecolor='white', linewidth=0.3)
            bottom += vals
    ax6.set_title('Severity by Type (Stacked)', color='white', fontweight='bold', pad=10)
    ax6.tick_params(colors='white', axis='y')
    ax6.tick_params(colors='white', axis='x', rotation=15, labelsize=8)
    ax6.spines[:].set_color('#30363D')
    ax6.legend(loc='upper right', framealpha=0.3,
               labelcolor='white', facecolor='#161B22', edgecolor='#30363D')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig('security_report_charts.png', dpi=150, bbox_inches='tight',
                facecolor='#0D1117', edgecolor='none')
    plt.show()
    print('\n📊 Charts saved to security_report_charts.png')


create_visualizations(df_vulns, risk_report)

## 📥 Section 11 — Print Vulnerability Report

In [ ]:
def print_vulnerability_report(df: pd.DataFrame, risk_report: dict, max_show: int = 20):
    """Pretty-print the top findings to the notebook."""
    if df.empty:
        print('✅ No vulnerabilities detected in this repository!')
        return

    SEV_EMOJI = {'Critical': '🔴', 'High': '🟠', 'Medium': '🟡', 'Low': '🟢'}

    print('\n' + '═'*70)
    print('  🔐 VULNERABILITY REPORT (Top findings)')
    print('═'*70)

    for i, row in df.head(max_show).iterrows():
        emoji = SEV_EMOJI.get(row['Severity'], '⚪')
        print(f"""
#{i+1} {emoji} [{row['Severity'].upper()}] {row['Vulnerability Name']}
   📁 File      : {row['File Name']}  (Line: {row['Line Number']})
   📌 Type      : {row['Vulnerability Type']}
   💡 Description: {row['Description'][:140]}...
   🤖 AI Insight : {row['AI Explanation'][:160]}...
   🔧 Fix        : {row['Recommended Fix'][:140]}
   🔗 Ref        : {row['Reference Link']}
   {'─'*66}""")

    if len(df) > max_show:
        print(f'\n   ... and {len(df) - max_show} more findings in the Excel report.')


print_vulnerability_report(df_vulns, risk_report)

## 📊 Section 12 — Excel Export

In [ ]:
def build_excel_report(df: pd.DataFrame, risk_report: dict, filename: str):
    """
    Create a professional, multi-sheet Excel report.
    Sheet 1: Executive Summary
    Sheet 2: Full Findings
    Sheet 3: Remediation Guide
    """

    wb = Workbook()

    # ─── Color palette ───────────────────────────────────────────────
    C = {
        'bg_dark'    : '0D1117',
        'bg_section' : '161B22',
        'accent_blue': '58A6FF',
        'white'      : 'FFFFFF',
        'grey_light' : 'F0F6FC',
        'grey_mid'   : '8B949E',
        'critical'   : 'FF0000',
        'high'       : 'E74C3C',
        'medium'     : 'F39C12',
        'low'        : '27AE60',
        'header_bg'  : '21262D',
        'row_alt'    : 'F8F9FA',
    }

    SEV_BG = {
        'Critical': 'FFEEF0',
        'High'    : 'FFF5F5',
        'Medium'  : 'FFFBF0',
        'Low'     : 'F0FFF4'
    }
    SEV_FG = {
        'Critical': 'B91C1C',
        'High'    : 'C0392B',
        'Medium'  : 'D97706',
        'Low'     : '15803D'
    }

    thin = Side(style='thin', color='D0D7DE')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    # ═══════════════════════════════════════════════════════════════
    # SHEET 1 — EXECUTIVE SUMMARY
    # ═══════════════════════════════════════════════════════════════
    ws1 = wb.active
    ws1.title = '📊 Executive Summary'
    ws1.sheet_view.showGridLines = False

    meta = risk_report['repo_meta']

    # Header banner
    ws1.merge_cells('A1:H1')
    ws1['A1'] = '🔐  GITHUB SECURITY VULNERABILITY REPORT'
    ws1['A1'].font = Font(bold=True, size=18, color=C['white'], name='Arial')
    ws1['A1'].fill = PatternFill('solid', fgColor=C['bg_dark'])
    ws1['A1'].alignment = Alignment(horizontal='center', vertical='center')
    ws1.row_dimensions[1].height = 40

    ws1.merge_cells('A2:H2')
    ws1['A2'] = f'Generated: {meta["scanned_at"]}  |  Powered by Gemini AI + OSV API'
    ws1['A2'].font = Font(size=10, color=C['grey_mid'], name='Arial', italic=True)
    ws1['A2'].fill = PatternFill('solid', fgColor=C['bg_section'])
    ws1['A2'].alignment = Alignment(horizontal='center')
    ws1.row_dimensions[2].height = 22

    # Repository info
    ws1.merge_cells('A4:H4')
    ws1['A4'] = '📌  REPOSITORY INFORMATION'
    ws1['A4'].font = Font(bold=True, size=12, color=C['white'], name='Arial')
    ws1['A4'].fill = PatternFill('solid', fgColor=C['header_bg'])
    ws1['A4'].alignment = Alignment(horizontal='left', indent=1)
    ws1.row_dimensions[4].height = 24

    repo_info = [
        ('Repository', meta['full_name']),
        ('URL', meta['url']),
        ('Language', meta['language']),
        ('Description', meta['description']),
        ('Stars', str(meta['stars'])),
        ('Forks', str(meta['forks'])),
        ('Repo Size', f"{meta['size_kb']:,} KB"),
    ]

    for idx, (label, value) in enumerate(repo_info, start=5):
        ws1[f'A{idx}'] = label
        ws1[f'A{idx}'].font = Font(bold=True, size=10, color=C['bg_dark'], name='Arial')
        ws1[f'A{idx}'].fill = PatternFill('solid', fgColor=C['grey_light'])
        ws1[f'A{idx}'].border = border
        ws1.merge_cells(f'B{idx}:H{idx}')
        ws1[f'B{idx}'] = value
        ws1[f'B{idx}'].font = Font(size=10, name='Arial')
        ws1[f'B{idx}'].border = border
        ws1[f'B{idx}'].alignment = Alignment(wrap_text=False)

    # Risk score block
    start_r = 13
    ws1.merge_cells(f'A{start_r}:H{start_r}')
    ws1[f'A{start_r}'] = '⚠️  RISK ASSESSMENT'
    ws1[f'A{start_r}'].font = Font(bold=True, size=12, color=C['white'], name='Arial')
    ws1[f'A{start_r}'].fill = PatternFill('solid', fgColor=C['header_bg'])
    ws1[f'A{start_r}'].alignment = Alignment(horizontal='left', indent=1)
    ws1.row_dimensions[start_r].height = 24

    cls_color_hex = risk_report['classification_color'].replace('#', '')
    risk_items = [
        ('Total Risk Score', str(risk_report['total_score'])),
        ('Risk Classification', risk_report['classification']),
        ('Total Vulnerabilities', str(len(df))),
        ('Critical', str(risk_report['counts'].get('Critical', 0))),
        ('High', str(risk_report['counts'].get('High', 0))),
        ('Medium', str(risk_report['counts'].get('Medium', 0))),
        ('Low', str(risk_report['counts'].get('Low', 0))),
    ]

    for idx, (label, value) in enumerate(risk_items, start=start_r + 1):
        ws1[f'A{idx}'] = label
        ws1[f'A{idx}'].font = Font(bold=True, size=10, color=C['bg_dark'], name='Arial')
        ws1[f'A{idx}'].fill = PatternFill('solid', fgColor=C['grey_light'])
        ws1[f'A{idx}'].border = border
        ws1.merge_cells(f'B{idx}:H{idx}')
        cell_val = ws1[f'B{idx}']
        cell_val.value = value
        cell_val.border = border
        cell_val.font = Font(size=10, bold=(label in ('Total Risk Score', 'Risk Classification')), name='Arial')
        # Color risk classification cell
        if label == 'Risk Classification':
            cell_val.fill = PatternFill('solid', fgColor=cls_color_hex)
            cell_val.font = Font(bold=True, size=11, color='FFFFFF', name='Arial')
        elif label in ('Critical', 'High', 'Medium', 'Low'):
            bg = SEV_BG.get(label, 'FFFFFF')
            cell_val.fill = PatternFill('solid', fgColor=bg.replace('#',''))
            cell_val.font = Font(bold=True, size=10,
                                  color=SEV_FG.get(label, '000000'), name='Arial')

    # ═══════════════════════════════════════════════════════════════
    # SHEET 2 — FULL FINDINGS
    # ═══════════════════════════════════════════════════════════════
    ws2 = wb.create_sheet('🔍 Full Findings')
    ws2.sheet_view.showGridLines = False
    ws2.freeze_panes = 'A3'

    headers = [
        'Repository Name', 'File Name', 'Line #', 'Type',
        'Vulnerability Name', 'Severity', 'Risk Score',
        'Description', 'AI Explanation', 'Recommended Fix',
        'Secure Code Example', 'Reference Link'
    ]
    col_widths = [22, 35, 8, 25, 38, 12, 10, 45, 55, 50, 50, 40]

    # Sheet title
    ws2.merge_cells(f'A1:{get_column_letter(len(headers))}1')
    ws2['A1'] = '🔍  FULL VULNERABILITY FINDINGS'
    ws2['A1'].font = Font(bold=True, size=14, color=C['white'], name='Arial')
    ws2['A1'].fill = PatternFill('solid', fgColor=C['bg_dark'])
    ws2['A1'].alignment = Alignment(horizontal='center', vertical='center')
    ws2.row_dimensions[1].height = 32

    # Column headers
    for col_idx, (header, width) in enumerate(zip(headers, col_widths), start=1):
        cell = ws2.cell(row=2, column=col_idx, value=header)
        cell.font = Font(bold=True, size=10, color=C['white'], name='Arial')
        cell.fill = PatternFill('solid', fgColor=C['header_bg'])
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = border
        ws2.column_dimensions[get_column_letter(col_idx)].width = width
    ws2.row_dimensions[2].height = 28

    # Data rows
    df_cols = [
        'Repository Name', 'File Name', 'Line Number', 'Vulnerability Type',
        'Vulnerability Name', 'Severity', 'Risk Score',
        'Description', 'AI Explanation', 'Recommended Fix',
        'Secure Code Example', 'Reference Link'
    ]

    for row_idx, row_data in enumerate(df[df_cols].itertuples(index=False), start=3):
        sev = row_data[5]  # Severity column
        row_bg = SEV_BG.get(sev, 'FFFFFF').replace('#', '')
        alt_bg = C['row_alt'] if row_idx % 2 == 0 else 'FFFFFF'

        for col_idx, value in enumerate(row_data, start=1):
            cell = ws2.cell(row=row_idx, column=col_idx, value=str(value) if value is not None else '')
            cell.font = Font(size=9, name='Arial')
            cell.border = border
            cell.alignment = Alignment(vertical='top', wrap_text=True)

            # Severity column special styling
            if col_idx == 6:  # Severity
                cell.fill = PatternFill('solid', fgColor=row_bg)
                cell.font = Font(bold=True, size=9,
                                  color=SEV_FG.get(sev, '000000'), name='Arial')
                cell.alignment = Alignment(horizontal='center', vertical='top')
            elif col_idx == 7:  # Risk Score
                cell.fill = PatternFill('solid', fgColor=row_bg)
                cell.font = Font(bold=True, size=9,
                                  color=SEV_FG.get(sev, '000000'), name='Arial')
                cell.alignment = Alignment(horizontal='center', vertical='top')
            else:
                cell.fill = PatternFill('solid', fgColor=alt_bg)

        ws2.row_dimensions[row_idx].height = 60

    # ═══════════════════════════════════════════════════════════════
    # SHEET 3 — REMEDIATION GUIDE
    # ═══════════════════════════════════════════════════════════════
    ws3 = wb.create_sheet('🔧 Remediation Guide')
    ws3.sheet_view.showGridLines = False

    ws3.merge_cells('A1:E1')
    ws3['A1'] = '🔧  REMEDIATION GUIDE — AI-Generated Secure Code & Fixes'
    ws3['A1'].font = Font(bold=True, size=14, color=C['white'], name='Arial')
    ws3['A1'].fill = PatternFill('solid', fgColor=C['bg_dark'])
    ws3['A1'].alignment = Alignment(horizontal='center', vertical='center')
    ws3.row_dimensions[1].height = 32

    rem_headers = ['Vulnerability Name', 'Severity', 'File', 'Recommended Fix', 'Secure Code Example']
    rem_widths  = [40, 12, 40, 60, 70]
    for col_idx, (h, w) in enumerate(zip(rem_headers, rem_widths), start=1):
        cell = ws3.cell(row=2, column=col_idx, value=h)
        cell.font = Font(bold=True, size=10, color=C['white'], name='Arial')
        cell.fill = PatternFill('solid', fgColor=C['header_bg'])
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = border
        ws3.column_dimensions[get_column_letter(col_idx)].width = w
    ws3.row_dimensions[2].height = 28

    rem_df = df[['Vulnerability Name', 'Severity', 'File Name', 'Recommended Fix', 'Secure Code Example']].drop_duplicates()
    for row_idx, row_data in enumerate(rem_df.itertuples(index=False), start=3):
        sev = row_data[1]
        alt_bg = C['row_alt'] if row_idx % 2 == 0 else 'FFFFFF'
        for col_idx, value in enumerate(row_data, start=1):
            cell = ws3.cell(row=row_idx, column=col_idx, value=str(value) if value else '')
            cell.font = Font(size=9, name='Courier New' if col_idx == 5 else 'Arial')
            cell.border = border
            cell.alignment = Alignment(vertical='top', wrap_text=True)
            if col_idx == 2:
                bg = SEV_BG.get(sev, 'FFFFFF').replace('#','')
                cell.fill = PatternFill('solid', fgColor=bg)
                cell.font = Font(bold=True, size=9,
                                  color=SEV_FG.get(sev,'000000'), name='Arial')
                cell.alignment = Alignment(horizontal='center', vertical='top')
            else:
                cell.fill = PatternFill('solid', fgColor=alt_bg)
        ws3.row_dimensions[row_idx].height = 80

    # ── Set column widths for Sheet 1 ────────────────────────────
    ws1.column_dimensions['A'].width = 24
    for col_letter in ['B','C','D','E','F','G','H']:
        ws1.column_dimensions[col_letter].width = 18

    wb.save(filename)
    print(f'\n✅ Excel report saved: {filename}')
    print(f'   📊 Sheets: {[ws.title for ws in wb.worksheets]}')
    print(f'   📄 Rows in findings: {len(df)}')
    return filename


# Export Excel
if not df_vulns.empty:
    excel_path = build_excel_report(df_vulns, risk_report, OUTPUT_EXCEL_FILE)
else:
    print('ℹ️  No vulnerabilities found — Excel report not generated.')

## 🎯 Section 13 — Final Dashboard Summary

In [ ]:
def print_final_dashboard(df: pd.DataFrame, risk_report: dict):
    """Print a rich terminal-style final summary dashboard."""
    meta = risk_report['repo_meta']
    cls  = risk_report['classification']

    EMOJI_MAP = {
        'Safe'         : '✅',
        'Moderate Risk': '⚠️ ',
        'High Risk'    : '🔶',
        'Critical Risk': '🔴'
    }

    print('\n' + '╔' + '═'*68 + '╗')
    print('║' + '  🔐  SECURITY SCAN FINAL DASHBOARD'.center(68) + '║')
    print('╠' + '═'*68 + '╣')
    print(f'║  Repository   : {meta["full_name"]:<51} ║')
    print(f'║  Scanned At   : {meta["scanned_at"]:<51} ║')
    print(f'║  Language     : {meta["language"]:<51} ║')
    print('╠' + '═'*68 + '╣')
    print(f'║  Total Vulnerabilities : {len(df):<43} ║')
    for sev in ['Critical', 'High', 'Medium', 'Low']:
        cnt = risk_report["counts"].get(sev, 0)
        print(f'║    {sev:<10}: {cnt:<55} ║')
    print('╠' + '═'*68 + '╣')
    print(f'║  Total Risk Score  : {risk_report["total_score"]:<48} ║')
    cls_str = f'{EMOJI_MAP.get(cls, "")} {cls}'
    print(f'║  Classification    : {cls_str:<48} ║')
    print('╠' + '═'*68 + '╣')
    print(f'║  📁 Excel Report   : {OUTPUT_EXCEL_FILE:<48} ║')
    print(f'║  📊 Charts Saved   : security_report_charts.png{" "*19} ║')
    print('╚' + '═'*68 + '╝')

    print('\n📋 NEXT STEPS:')
    print('  1. Open the Excel report for detailed findings + AI remediation')
    print('  2. Prioritize Critical → High vulnerabilities first')
  
    if risk_report['counts'].get('Critical', 0) > 0:
        print('  3. 🚨 CRITICAL issues found — immediately revoke any exposed credentials!')
    if risk_report['counts'].get('High', 0) > 0:
        print('  4. Address HIGH severity insecure code patterns before next deployment')
    print('  5. Implement pre-commit hooks + CI/CD security scanning to prevent regressions')
    print('  6. Review the "Secure Code Example" column in the Remediation Guide sheet')


print_final_dashboard(df_vulns, risk_report)

## 🔄 Section 14 — Multi-Repository Scanner (Bonus Feature)

In [ ]:
def scan_multiple_repos(repo_urls: list, github_token: str = '', gemini_key: str = ''):
    """
    Bonus feature: scan multiple repositories and produce a combined report.
    Usage:
        scan_multiple_repos([
            'https://github.com/user/repo1',
            'https://github.com/user/repo2',
        ])
    """
    g     = authenticate_github(github_token)
    model = configure_gemini(gemini_key)
    all_frames   = []
    risk_reports = []

    for url in repo_urls:
        print(f'\n{'─'*60}')
        print(f'  Scanning: {url}')
        try:
            rdata   = fetch_repository_files(g, parse_repo_url(url))
            df_scan = run_full_scan(rdata, model)
            rpt     = calculate_risk_score(df_scan, rdata['repo_meta'])
            all_frames.append(df_scan)
            risk_reports.append(rpt)
        except Exception as e:
            print(f'  ❌ Failed: {e}')

    if not all_frames:
        print('No results to combine.')
        return

    combined_df = pd.concat(all_frames, ignore_index=True)

    # Combined risk summary
    print('\n\n📊 MULTI-REPO SCAN SUMMARY')
    print('─'*60)
    for rpt in risk_reports:
        m = rpt['repo_meta']
        print(f"  {m['full_name']:<40} | {rpt['classification']:<16} | Score: {rpt['total_score']}")
    print(f"\n  Combined: {len(combined_df)} total vulnerabilities across {len(risk_reports)} repos")

    out_file = 'multi_repo_security_report.xlsx'
    combined_risk = calculate_risk_score(combined_df, {
        'full_name' : 'Multi-Repo Scan', 'url': '', 'language': 'Mixed',
        'description': 'Combined scan', 'stars': 0, 'forks': 0,
        'size_kb': 0, 'scanned_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    })
    build_excel_report(combined_df, combined_risk, out_file)
    print(f'  💾 Combined report: {out_file}')


# ── Example (uncomment to use): ─────────────────────────────────
# scan_multiple_repos([
#     'https://github.com/OWASP/WebGoat',
#     'https://github.com/bridgecrewio/terragoat',
# ], github_token=GITHUB_TOKEN, gemini_key=GEMINI_API_KEY)

print('✅ Multi-repo scanner ready.')
print('   Uncomment the scan_multiple_repos() call above to use it.')

## 📚 Section 15 — Quick Reference & Instructions

---

### 🚀 How to Run This Notebook

1. **Get Free API Keys:**
   - **Gemini** (AI analysis): https://aistudio.google.com/app/apikey → free, generous quota
   - **GitHub Token** (optional, higher rate limits): https://github.com/settings/tokens

2. **Edit Section 3** — fill in your keys:
   ```python
   GITHUB_REPO_URL = 'https://github.com/your/target-repo'
   GITHUB_TOKEN    = 'ghp_xxxxxxxxxxxxx'   # optional
   GEMINI_API_KEY  = 'AIzaSyxxxxxxxxxxxxxxxxx'  # recommended
   ```

3. **Run All Cells** (Kernel → Restart & Run All)

4. **View outputs:**
   - Printed report in Section 11
   - Charts in Section 10
   - Excel file: `github_security_vulnerability_report.xlsx`

---

### 🧪 Recommended Test Repositories (contain intentional vulnerabilities)

| Repository | Description |
|---|---|
| `https://github.com/OWASP/WebGoat` | OWASP deliberately insecure Java app |
| `https://github.com/bridgecrewio/terragoat` | Insecure Terraform code |
| `https://github.com/payatu/vuln-webgoat` | Python vulnerable web app |
| `https://github.com/cider-security-research/cicd-goat` | CI/CD vulnerabilities |

---

### ⚙️ Configuration Options

| Setting | Default | Description |
|---|---|---|
| `MAX_FILES_TO_SCAN` | 100 | Cap files per scan (increase for thorough scans) |
| `AI_DELAY_SECONDS` | 2 | Delay between Gemini calls (reduce if using paid tier) |
| `OUTPUT_EXCEL_FILE` | `github_security_vulnerability_report.xlsx` | Output filename |

---

### 📊 Risk Classification System

| Score | Classification | Action |
|---|---|---|
| 0–20 | ✅ Safe | No urgent action required |
| 21–60 | ⚠️ Moderate Risk | Fix within sprint |
| 61–120 | 🔶 High Risk | Fix within days |
| 121+ | 🔴 Critical Risk | Immediate action — potential breach risk |